In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


from typing import Optional  # change Optional[str] to str | None when using higher versions of python3

%matplotlib inline

## data path - change this 
path_to_github_folder = "/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/"
path_to_plotting_folder = path_to_github_folder + "plottingCode/"
path_to_file_folder =  "fc_data/Data_formation_channels_intrinsic/"
path_to_file = path_to_github_folder + path_to_file_folder
path_to_default_scripts = path_to_plotting_folder
path_to_save_fig = path_to_github_folder + "figures/"
sys.path.append(path_to_plotting_folder +'.')

from default_scripts import *  # imports standard functions for formatting and for dictionary of colors used

## Create a zip file folder that contains the PNG images to create the large legend for the powerpoint overview of formationchannels


In [2]:
import os
import re
from collections import defaultdict

import matplotlib as mpl
mpl.use("Agg")  # headless / no GUI

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def sanitize_filename(text: str, max_len: int = 140) -> str:
    text = text.strip()
    text = re.sub(r"\s+", "_", text)
    text = text.replace("\\", "")  # avoid awkward filenames
    text = text.replace("{", "").replace("}", "")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = text.strip("._-")
    return (text or "label")[:max_len]


def save_patch_png(
    color: str,
    hatch, #hatch: str | None,
    outpath: str,
    *,
    width_px: int = 420,
    height_px: int = 120,
    dpi: int = 200,
    edgecolor: str = "black",
    linewidth: float = 0.8,
    hatch_color: str = "darkslategray",
    hatch_linewidth: float = 0.6,
    transparent_bg: bool = True,
    pad_fraction: float = 0.12,
):
    """
    Render a single Matplotlib Patch (with hatch) to a PNG, tightly cropped.
    """
    # Hatch tuning (matches your style intent)
    mpl.rcParams["hatch.linewidth"] = hatch_linewidth
    mpl.rcParams["hatch.color"] = hatch_color

    # Compute a figure size that roughly matches your requested pixels
    fig_w = width_px / dpi
    fig_h = height_px / dpi

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    pad = pad_fraction
    rect = mpatches.Rectangle(
        (pad, pad),
        1 - 2 * pad,
        1 - 2 * pad,
        facecolor=color,
        edgecolor=edgecolor,
        linewidth=linewidth,
        hatch=(hatch if hatch else ""),
    )
    ax.add_patch(rect)

    fig.savefig(
        outpath,
        format="png",
        dpi=dpi,
        transparent=transparent_bg,
        bbox_inches="tight",
        pad_inches=0.0,
    )
    plt.close(fig)


def export_channel_patches_as_pngs(
    fc_dict: dict,
    out_dir: str = "legend_color_bars_mpl",
    *,
    width_px: int = 420,
    height_px: int = 120,
    dpi: int = 200,
    transparent_bg: bool = True,
):
    """
    Writes one PNG per unique (label, color, hatch). If a label maps to multiple styles,
    filenames get suffixes _01, _02, etc.
    """
    os.makedirs(out_dir, exist_ok=True)

    # label -> set of (color, hatch)
    label_to_styles = defaultdict(set)

    for group in fc_dict.values():
        if isinstance(group, dict) and "channels" in group:
            for ch in group["channels"]:
                label = ch.get("label")
                color = ch.get("color")
                hatch = ch.get("hatch")
                if label and color:
                    label_to_styles[label].add((color, hatch))
        elif isinstance(group, dict):
            # e.g. total; you can include/exclude as you wish
            label = group.get("label")
            color = group.get("color")
            hatch = group.get("hatch")
            if label and color:
                label_to_styles[label].add((color, hatch))

    n_written = 0
    for label, styles in sorted(label_to_styles.items(), key=lambda x: x[0].lower()):
        base = sanitize_filename(label)
        styles = sorted(styles)

        if len(styles) == 1:
            color, hatch = styles[0]
            outpath = os.path.join(out_dir, f"{base}.png")
            save_patch_png(
                color, hatch, outpath,
                width_px=width_px, height_px=height_px, dpi=dpi,
                transparent_bg=transparent_bg,
            )
            n_written += 1
        else:
            for i, (color, hatch) in enumerate(styles, start=1):
                outpath = os.path.join(out_dir, f"{base}_{i:02d}.png")
                save_patch_png(
                    color, hatch, outpath,
                    width_px=width_px, height_px=height_px, dpi=dpi,
                    transparent_bg=transparent_bg,
                )
                n_written += 1

    print(f"Wrote {n_written} PNG(s) to: {os.path.abspath(out_dir)}")


# -----------------------
# Usage
# -----------------------
fc_dict = obtain_fc_dict()
outpath_png = path_to_save_fig + "legend_color_bars_mpl"
export_channel_patches_as_pngs(fc_dict, out_dir=outpath_png, transparent_bg=True)


Wrote 20 PNG(s) to: /Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/figures/legend_color_bars_mpl
